In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

In [ ]:
df = pd.read_csv("clientes_frutas.csv")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_cliente                    2000 non-null   int64  
 1   edad                          2000 non-null   int64  
 2   ciudad                        2000 non-null   object 
 3   tipo_cliente                  2000 non-null   object 
 4   dias_registrado               2000 non-null   int64  
 5   compras_totales               2000 non-null   int64  
 6   compras_ultimos_90_dias       2000 non-null   int64  
 7   gasto_total_90_dias           2000 non-null   float64
 8   promedio_compra               2000 non-null   float64
 9   dias_desde_ultima_compra      2000 non-null   int64  
 10  frecuencia_compra_dias        2000 non-null   int64  
 11  regularidad_compra            2000 non-null   float64
 12  porcentaje_descuentos_usados  2000 non-null   float64
 13  usa

In [ ]:
# La ID no es importante
df = df.drop(columns="id_cliente")

In [ ]:
# Codificación de variables categóricas
df_prepared = pd.get_dummies(df, columns=['ciudad', 'tipo_cliente'], drop_first=True)

In [ ]:
# Conteo de clientes clasificados (deben balancearse)
df['abandono'].value_counts()

,count
abandono,
0,1396
1,604


In [ ]:
# Separar variables predictoras (X) y objetivo (y)
X = df_prepared.drop('abandono', axis=1)
y = df_prepared['abandono']

# División de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [ ]:
# Balanceo de datos (604 vs 1396)
smt = SMOTE(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

print(f"Distribución original: {y_train.value_counts().to_dict()}")
print(f"Distribución balanceada: {y_train_res.value_counts().to_dict()}")

In [ ]:
# Entrenamiento del modelo Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Evaluación
y_pred = rf_model.predict(X_test)

Distribución original: {0: 977, 1: 423}
Distribución balanceada: {0: 977, 1: 977}

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       419
           1       0.98      0.92      0.95       181

    accuracy                           0.97       600
   macro avg       0.97      0.96      0.96       600
weighted avg       0.97      0.97      0.97       600


Principales factores de abandono:
compras_ultimos_90_dias         0.353173
gasto_total_90_dias             0.208694
dias_desde_ultima_compra        0.192607
compras_totales                 0.050199
porcentaje_descuentos_usados    0.036284
dtype: float64


In [ ]:
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

# Importancia de las variables
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nPrincipales factores de abandono:")
print(importances.head(5))

### Guardar el Modelo y Función de Predicción
Ahora guardaremos el modelo entrenado y definiremos una función para facilitar su uso con datos nuevos.

In [ ]:
class AbandonoPredictor:
    def __init__(self, model_path='modelo_abandono.joblib', columns_path='columnas_modelo.joblib'):
        """Inicializa la clase cargando el modelo y las columnas necesarias."""
        try:
            self.model = joblib.load(model_path)
            self.model_columns = joblib.load(columns_path)
            print("Modelo y columnas cargados exitosamente.")
        except Exception as e:
            print(f"Error al cargar el modelo: {e}")

    def _preprocesar(self, datos_cliente):
        """Método privado para preparar los datos antes de la predicción."""
        df_nuevo = pd.DataFrame([datos_cliente])
        # Codificación One-Hot
        df_nuevo_prepared = pd.get_dummies(df_nuevo)

        # Alineación de columnas con el set de entrenamiento
        for col in self.model_columns:
            if col not in df_nuevo_prepared.columns:
                df_nuevo_prepared[col] = 0

        return df_nuevo_prepared[self.model_columns]

    def predecir(self, datos_cliente):
        """Realiza la predicción y devuelve un informe detallado."""
        df_final = self._preprocesar(datos_cliente)

        prediccion = self.model.predict(df_final)[0]
        probabilidad = self.model.predict_proba(df_final)[0][1]

        estado = "RIESGO DE ABANDONO" if prediccion == 1 else "CLIENTE LEAL"

        return {
            'resultado': estado,
            'probabilidad_abandono': f"{probabilidad:.2%}",
            'score': probabilidad
        }

#### Ejemplo de uso con un nuevo cliente:

In [ ]:
# Cliente con pocas compras recientes
nuevo_cliente = {
    'edad': 45,
    'ciudad': 'Manta',
    'tipo_cliente': 'Minorista',
    'dias_registrado': 500,
    'compras_totales': 10,
    'compras_ultimos_90_dias': 0,
    'gasto_total_90_dias': 0.0,
    'promedio_compra': 50.0,
    'dias_desde_ultima_compra': 85,
    'frecuencia_compra_dias': 30,
    'regularidad_compra': 0.1,
    'porcentaje_descuentos_usados': 0.05,
    'usa_domicilio': 1,
    'reclamos': 2,
    'calificacion_promedio': 3.5
}

#### Ejemplo de uso con la nueva clase:

In [ ]:
# Instanciar el predictor
predictor = AbandonoPredictor()
informe = predictor.predecir(nuevo_cliente)

# resultados
print(f"\n--- Informe del Cliente ---")
print(f"Estado: {informe['resultado']}")
print(f"Probabilidad: {informe['probabilidad_abandono']}")

Modelo y columnas cargados exitosamente.

--- Informe del Cliente ---
Estado: RIESGO DE ABANDONO
Probabilidad: 88.00%
